# S11 · Find the groups

We have a pile of points with no labels, and we want to find the groups hiding in
them. First we use **k-means** on round blobs and learn two honest ways to decide
how many groups to ask for. Then we use **DBSCAN** on crescent shapes that k-means
gets wrong. This is the shop-customer idea from the session: dots that sit close
together are similar, and clustering finds the clumps.

**New here? Read this once.**

- New to Python? You can still do this whole notebook. Press the play button on
  each cell, top to bottom, and read the plain-English note above each one.
- New to the idea of "how far apart two dots are"? Open the primer
  `primers/distance_between_points.md` for a ten-minute, picture-first version.
- Already confident with code or with clustering? Skip ahead to the cells marked
  **Stretch (optional)**.
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. On your **own machine** you already
installed everything with `uv`, so it does nothing there.

In [ ]:
# This notebook uses numpy, matplotlib and scikit-learn.
# Google Colab already ships all three, so there is nothing to install.
print("Setup complete - nothing to install.")

These are all the tools this notebook needs. We import them once, here, so the
rest of the notebook stays about the ideas.

In [ ]:
import numpy as np                          # fast maths on lists of numbers
import matplotlib.pyplot as plt              # drawing charts
from sklearn.datasets import make_blobs      # makes round clouds of points to play with
from sklearn.datasets import make_moons      # makes two crescent shapes
from sklearn.cluster import KMeans           # the k-means clustering tool
from sklearn.cluster import DBSCAN           # the density-based clustering tool
from sklearn.metrics import silhouette_score # scores how well the groups came out

## Step 1 — make some points to group

Unsupervised learning means the data has **no answer column**. We just have points,
and we want to find the groups hiding in them.

`make_blobs` hands us a few round clouds of points. Picture each dot as one customer
placed by two of their numbers. We secretly ask for 3 clouds so we know the true
answer, but the clustering tool will never be told that number.

In [ ]:
# Set a seed so we all get exactly the same random points every time.
np.random.seed(0)

# Make 300 points arranged in 3 round blobs.
# points  : the (x, y) location of each point  -> shape (300, 2)
# true_id : which blob each point really came from (we will mostly ignore this)
points, true_id = make_blobs(
    n_samples=300,
    centers=3,
    cluster_std=0.9,
    random_state=0)

print("points shape:", points.shape)
print("first 3 points:")
print(points[:3].round(2))

## Step 2 — look at the data first

Always plot your data before doing anything. We draw all the points in one colour,
because we are pretending we do not yet know the groups. Ask yourself: how many
clumps do you see?

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(points[:, 0], points[:, 1], color="#7F7F7F", s=20)
plt.xlabel("feature 1")
plt.ylabel("feature 2")
plt.title("The raw points (no labels yet) - how many groups do you see?")
plt.show()

## Step 3 — let k-means find the groups

Now the one idea of today. K-means groups points by **distance**: each point joins
the nearest group centre, which is called a **centroid**. We tell it how many groups
to look for with `n_clusters`. We saw three clumps above, so we ask for 3.

In [ ]:
# n_clusters = how many groups we ask for.
# random_state makes the result repeatable.
# n_init=10 means it tries 10 random starts and keeps the best one.
kmeans = KMeans(n_clusters=3, random_state=0, n_init=10)

# fit_predict does two things: it finds the groups, and it returns a label
# (0, 1 or 2) for every point saying which group it landed in.
kmeans_labels = kmeans.fit_predict(points)

print("first 20 group labels:", kmeans_labels[:20])

# The centroids: the centre point of each group.
centroids = kmeans.cluster_centers_
print()
print("centroids (group centres):")
print(centroids.round(2))

## Step 4 — plot the groups and their centres

Now we colour each point by the group k-means gave it, and draw the centroids as
black crosses. The colours should match the clumps you saw above.

In [ ]:
plt.figure(figsize=(7, 5))

# Colour the points by their k-means group label.
plt.scatter(points[:, 0], points[:, 1], c=kmeans_labels, cmap="viridis", s=20)

# Draw the centroids on top, as big black X marks.
plt.scatter(centroids[:, 0], centroids[:, 1],
            color="black", marker="X", s=200, label="centroids")

plt.xlabel("feature 1")
plt.ylabel("feature 2")
plt.title("K-means groups, with the centroids marked")
plt.legend()
plt.show()

## Step 5 — but how many groups should we ask for?

We told k-means to find 3 groups, but in real life nobody hands you the number. The
**elbow method** helps. For each choice of k we measure the **inertia**: the total
squared distance from every point to its own centroid. Smaller inertia means tighter,
neater groups.

In [ ]:
# We will try k = 1, 2, 3, ... up to 8 and record the inertia each time.
k_values = range(1, 9)
inertia_values = []

for k in k_values:
    model = KMeans(n_clusters=k, random_state=0, n_init=10)
    model.fit(points)
    # inertia_ is the total squared distance of points to their centroid.
    inertia_values.append(model.inertia_)
    print("k =", k, " inertia =", round(model.inertia_, 1))

## Step 6 — draw the elbow

Inertia always falls as k grows, because more groups means every point is nearer some
centre. So we do not want the smallest value. We want the **bend**, the point where
adding another group stops helping much. That bend is the elbow.

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(list(k_values), inertia_values, "-o", color="#2E75B6")
plt.xlabel("number of groups  k")
plt.ylabel("inertia (smaller = tighter groups)")
plt.title("Elbow method: the bend suggests the right k")
plt.show()

print("The curve bends sharply at k = 3, which matches the 3 blobs we made.")

## Step 7 — a second opinion: the silhouette score

The **silhouette score** is one number between -1 and +1 that says how well the points
sit in their groups: close to their own group, far from the others. Higher is better.
We compute it for a few values of k and pick the highest.

In [ ]:
# The silhouette needs at least 2 groups, so we start at k = 2.
for k in range(2, 7):
    model = KMeans(n_clusters=k, random_state=0, n_init=10)
    labels = model.fit_predict(points)
    score = silhouette_score(points, labels)
    print("k =", k, " silhouette score =", round(score, 3))

print()
print("The highest score is at k = 3 - the elbow and the silhouette agree.")

### Stretch (optional) — do one k-means round by hand

Skip this if you are new to code. If you want to see what k-means actually does
inside, here is one full round with three chosen starting centres. We measure each
point's distance to each centre, give every point to its nearest centre, then move
each centre to the average of the points that chose it. That "assign, then move" is
the whole algorithm, repeated until nothing changes.

In [ ]:
# Three starting centres we picked by hand (k-means would pick these randomly).
starting_centres = np.array([[-2.0, 2.0], [0.0, 0.0], [2.0, -2.0]])

# Distance from every point to every centre.
# points[:, None, :] - starting_centres has shape (300, 3, 2): for each of the 300
# points, the gap to each of the 3 centres. We take the length of each gap.
gaps = points[:, None, :] - starting_centres[None, :, :]
distances = np.sqrt((gaps ** 2).sum(axis=2))          # shape (300, 3)

# Each point joins its nearest centre: 0, 1 or 2.
nearest_centre = distances.argmin(axis=1)

# Move each centre to the average of the points that chose it.
new_centres = np.zeros_like(starting_centres)
for group_number in range(3):
    points_in_group = points[nearest_centre == group_number]
    new_centres[group_number] = points_in_group.mean(axis=0)

print("started at centres:")
print(starting_centres)
print()
print("after one assign-and-move round, centres are:")
print(new_centres.round(2))
print()
print("Run this idea a few more times and the centres stop moving. That is k-means.")

## Step 8 — now make crescent shapes, where k-means struggles

K-means always carves the space into round, straight-edged regions around its
centres. So it fails on shapes that are not round blobs. `make_moons` gives two
interleaving crescents, a classic hard case.

In [ ]:
# Two crescent ("moon") shapes with a little random wobble.
moon_points, moon_true_id = make_moons(
    n_samples=300,
    noise=0.06,
    random_state=0)

plt.figure(figsize=(7, 5))
plt.scatter(moon_points[:, 0], moon_points[:, 1], color="#7F7F7F", s=20)
plt.xlabel("feature 1")
plt.ylabel("feature 2")
plt.title("Two crescents - clearly two groups, but not round blobs")
plt.show()

## Step 9 — k-means on the crescents (it gets them wrong)

We ask k-means for 2 groups. Watch how it slices straight through the crescents
instead of following their curved shape.

In [ ]:
kmeans_moons = KMeans(n_clusters=2, random_state=0, n_init=10)
kmeans_moon_labels = kmeans_moons.fit_predict(moon_points)

plt.figure(figsize=(7, 5))
plt.scatter(moon_points[:, 0], moon_points[:, 1],
            c=kmeans_moon_labels, cmap="viridis", s=20)
plt.xlabel("feature 1")
plt.ylabel("feature 2")
plt.title("K-means cuts straight through the crescents - wrong groups")
plt.show()

## Step 10 — DBSCAN follows the shape

DBSCAN groups points by **crowding** instead of distance to a centre: wherever points
are packed close together they form a group, and the group can be any shape. It also
labels lonely points as **noise** (given the label -1), which is handy for spotting
outliers.

It has two settings:

- `eps`: how close two points must be to count as neighbours.
- `min_samples`: how many neighbours a point needs to count as being "in a crowd".

In [ ]:
# eps and min_samples chosen to suit the spacing of the moon points.
dbscan = DBSCAN(eps=0.2, min_samples=5)
dbscan_labels = dbscan.fit_predict(moon_points)

# The label -1 means "noise" - a point in no crowd.
print("group labels found:", np.unique(dbscan_labels))
number_of_noise_points = np.sum(dbscan_labels == -1)
print("number of noise points:", number_of_noise_points)

## Step 11 — plot DBSCAN's result

DBSCAN should recover the two curved crescents that k-means could not. Any noise
points (label -1) are drawn in grey as small crosses.

In [ ]:
plt.figure(figsize=(7, 5))

# Split into noise points and real cluster points so we can colour them apart.
is_noise = dbscan_labels == -1

# Real clusters, coloured by their label.
plt.scatter(moon_points[~is_noise, 0], moon_points[~is_noise, 1],
            c=dbscan_labels[~is_noise], cmap="viridis", s=20)

# Noise points in grey, drawn as x marks.
plt.scatter(moon_points[is_noise, 0], moon_points[is_noise, 1],
            color="#BBBBBB", marker="x", s=40, label="noise")

plt.xlabel("feature 1")
plt.ylabel("feature 2")
plt.title("DBSCAN follows the crowding and recovers the two crescents")
plt.legend()
plt.show()

## What you just did

You ran two clustering methods on **unlabelled** data. K-means is fast and great for
round blobs, but you must choose the number of groups yourself, and the elbow and
silhouette help you choose it honestly. DBSCAN needs no number up front, follows any
shape, and flags outliers as noise. Both rest on the same idea: grouping by how close
points are.

Next notebook: `02_squash_to_two_columns.ipynb`, where we take data with too many
columns to plot and squash it down to two, so we can actually see groups like these.